# Multi-Query Attention（MQA）

源码导航：[core/attention/mqa.py](../../../core/attention/mqa.py) 中的 `MQACausalSelfAttention`。

MQA 是 GQA 的极端情形：**所有 Q 头共享唯一一组 K/V 头**（`n_head_kv = 1`）。Google PaLM 率先在大规模模型中采用，将 KV cache 从 $O(n_{\text{head}} \cdot T \cdot d_h)$ 压缩到 $O(T \cdot d_h)$。

### 1. 理论推导

| 机制 | Q 头数 | KV 头数 | KV cache（每层每 token） |
|---|---|---|---|
| MHA | $H$ | $H$ | $2 H d_h$ |
| GQA | $H$ | $G \ll H$ | $2 G d_h$ |
| **MQA** | $H$ | **1** | **$2 d_h$** |

MQA 通过 `repeat_kv` 将单组 KV 广播到所有 Q 头，注意力计算形式不变。

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import torch

ROOT = Path.cwd()
while ROOT.name and not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from core.attention.mqa import MQACausalSelfAttention
from core.attention.walkie_attention import WalkieCausalSelfAttention, repeat_kv

### 2. KV 头广播与形状检查

In [ ]:
torch.manual_seed(0)
B, T, C, H = 2, 16, 64, 8
x = torch.randn(B, T, C)

mqa = MQACausalSelfAttention(n_embd=C, n_head=H, attn_impl='eager')
out = mqa(x)

assert out.shape == (B, T, C)
assert mqa.n_head_kv == 1
assert mqa.n_rep == H

kv = torch.randn(B, 1, T, mqa.head_dim)
expanded = repeat_kv(kv, n_rep=H)
print('KV 单头:', tuple(kv.shape))
print('广播后:', tuple(expanded.shape))
print('MQA output:', tuple(out.shape))

### 3. 与 GQA 的 KV cache 对比

In [ ]:
def kv_cache_elems(n_head_kv: int, head_dim: int) -> int:
    return 2 * n_head_kv * head_dim  # K + V per token per layer

H, d = 8, 64
print('MHA cache:', kv_cache_elems(H, d))
print('GQA (4 KV):', kv_cache_elems(4, d))
print('MQA cache:', kv_cache_elems(1, d))

### 4. 源码精讲

`MQACausalSelfAttention` 继承 `WalkieCausalSelfAttention`，在 `__init__` 中固定 `n_head_kv=1`，其余 QK-Norm、RoPE、`attn_impl` 路径与 GQA 完全一致。

---

## 延伸阅读与参考资料

- Ainslie et al., *GQA: Training Generalized Multi-Query Transformer Models from Multi-Head Checkpoints* (2023). [arXiv:2305.13245](https://arxiv.org/abs/2305.13245)
- Chowdhery et al., *PaLM* (2022). MQA 大规模实践。